# Sutton & Barto Chapter 2: 2.6-2.7
## Optimistic Initial Values and UCB

This notebook implements two classic exploration ideas for the **10-armed testbed**:

- **2.6 Optimistic Initial Values**
- **2.7 Upper-Confidence-Bound (UCB) Action Selection**

Both methods encourage exploration, but they do so in different ways.

## What You Will See

- greedy action selection with optimistic initialization
- the UCB exploration bonus from Section 2.7
- a direct comparison against standard $\epsilon$-greedy
- small, self-contained code suitable for study and reuse

Dependencies: `numpy`, `matplotlib`

## Theory

### Optimistic initial values
If we initialize all estimates with a large value, each action looks promising at the start. A greedy agent then explores because disappointing rewards pull estimates down over time.

### UCB action selection
UCB adds an explicit uncertainty bonus:

$$
A_t = \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right].
$$

The first term prefers high estimated value, while the second term prefers less-visited actions. As counts increase, the exploration bonus shrinks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

In [ ]:
class TenArmedBandit:
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_true = self.rng.normal(0.0, 1.0, size=k)

    def step(self, action):
        reward = self.rng.normal(self.q_true[action], 1.0)
        optimal = int(action == np.argmax(self.q_true))
        return reward, optimal

In [ ]:
def run_greedy_optimistic(steps=1000, initial_value=5.0, alpha=0.1, seed=None):
    bandit = TenArmedBandit(seed=seed)
    rng = np.random.default_rng(seed)
    q_est = np.full(bandit.k, initial_value, dtype=float)
    rewards = np.zeros(steps)
    optimal_actions = np.zeros(steps)

    for t in range(steps):
        action = rng.choice(np.flatnonzero(q_est == q_est.max()))
        reward, optimal = bandit.step(action)
        q_est[action] += alpha * (reward - q_est[action])
        rewards[t] = reward
        optimal_actions[t] = optimal

    return rewards, optimal_actions


def run_ucb(steps=1000, c=2.0, seed=None):
    bandit = TenArmedBandit(seed=seed)
    rng = np.random.default_rng(seed)
    q_est = np.zeros(bandit.k)
    counts = np.zeros(bandit.k, dtype=int)
    rewards = np.zeros(steps)
    optimal_actions = np.zeros(steps)

    for t in range(steps):
        bonus = np.where(
            counts > 0,
            c * np.sqrt(np.log(t + 1 + 1e-9) / counts),
            np.inf,
        )
        action = rng.choice(np.flatnonzero(q_est + bonus == np.max(q_est + bonus)))
        reward, optimal = bandit.step(action)
        counts[action] += 1
        q_est[action] += (reward - q_est[action]) / counts[action]
        rewards[t] = reward
        optimal_actions[t] = optimal

    return rewards, optimal_actions


def run_epsilon_greedy(steps=1000, epsilon=0.1, seed=None):
    bandit = TenArmedBandit(seed=seed)
    rng = np.random.default_rng(seed)
    q_est = np.zeros(bandit.k)
    counts = np.zeros(bandit.k, dtype=int)
    rewards = np.zeros(steps)
    optimal_actions = np.zeros(steps)

    for t in range(steps):
        if rng.random() < epsilon:
            action = rng.integers(bandit.k)
        else:
            action = rng.choice(np.flatnonzero(q_est == q_est.max()))
        reward, optimal = bandit.step(action)
        counts[action] += 1
        q_est[action] += (reward - q_est[action]) / counts[action]
        rewards[t] = reward
        optimal_actions[t] = optimal

    return rewards, optimal_actions


def average_runs(runner, runs=200, **kwargs):
    reward_history = []
    optimal_history = []
    for seed in range(runs):
        rewards, optimal = runner(seed=seed, **kwargs)
        reward_history.append(rewards)
        optimal_history.append(optimal)
    return np.mean(reward_history, axis=0), 100 * np.mean(optimal_history, axis=0)

## 2.6 Optimistic initial values

A purely greedy method usually gets stuck if all estimates start at zero. But if all initial estimates are **too high**, then every action looks worth trying.

Below we compare:

- realistic initialization: `Q_1(a)=0`
- optimistic initialization: `Q_1(a)=5`


In [ ]:
reward_eg, opt_eg = average_runs(run_epsilon_greedy, runs=200, steps=1000, epsilon=0.1)
reward_opt, opt_opt = average_runs(run_greedy_optimistic, runs=200, steps=1000, initial_value=5.0, alpha=0.1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(reward_eg, label='epsilon-greedy (epsilon=0.1)')
ax[0].plot(reward_opt, label='greedy with optimistic Q1=5')
ax[0].set_title('Average Reward')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Average reward')
ax[0].legend()

ax[1].plot(opt_eg, label='epsilon-greedy (epsilon=0.1)')
ax[1].plot(opt_opt, label='greedy with optimistic Q1=5')
ax[1].set_title('% Optimal Action')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('% optimal action')
ax[1].legend()
plt.tight_layout()

## 2.7 UCB action selection

UCB makes exploration more principled than random exploration. Instead of trying actions with a fixed probability, it asks which action has the best combination of:

- high estimated reward
- high uncertainty because it has not been selected often


In [ ]:
reward_ucb, opt_ucb = average_runs(run_ucb, runs=200, steps=1000, c=2.0)
reward_eg2, opt_eg2 = average_runs(run_epsilon_greedy, runs=200, steps=1000, epsilon=0.1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(reward_eg2, label='epsilon-greedy (epsilon=0.1)')
ax[0].plot(reward_ucb, label='UCB (c=2)')
ax[0].set_title('Average Reward')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Average reward')
ax[0].legend()

ax[1].plot(opt_eg2, label='epsilon-greedy (epsilon=0.1)')
ax[1].plot(opt_ucb, label='UCB (c=2)')
ax[1].set_title('% Optimal Action')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('% optimal action')
ax[1].legend()
plt.tight_layout()

## Takeaway

- Optimistic initial values create **temporary exploration** even with a greedy policy.
- UCB adds an explicit confidence bonus and often explores more efficiently than plain $\epsilon$-greedy.
- Both are simple ways to improve exploration in the stationary 10-armed testbed.

## Reference

Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, Chapter 2.